upgradeing the miner to pull the number of the changes to each style in dwelling period

# Step 1: Clone

In [ ]:
from __future__ import annotations
import csv, os, re, subprocess, time, base64
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from urllib.parse import urlparse

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT       = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3")
URL_LIST_CSV    = WORK_ROOT / "URL_List_RQ3.csv"
CLONE_ROOT      = WORK_ROOT / "Clone"
MANIFEST_CSV    = WORK_ROOT / "clones_manifest_V1_retry.csv"

# Token file + keys (you said you already have these)
TOKENS_ENV_FILE = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
TOKEN_KEYS      = [f"GITHUB_TOKEN_{i}" for i in range(1, 7)]

# Optional: keep your PR refs option
FETCH_PR_REFS   = True

# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
DEFAULT_TIMEOUT = 1800  # 30 minutes for huge repos
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}

def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}")

def load_env_file(path: Path) -> None:
    """
    Loads KEY=VALUE lines from a .env-like file into os.environ (non-destructive).
    Supports lines like:
      GITHUB_TOKEN_1=...
      export GITHUB_TOKEN_2="..."
      # comments
    """
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path}")
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.lower().startswith("export "):
            line = line[7:].strip()
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and v and k not in os.environ:
            os.environ[k] = v

def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    # SSH form: git@host:owner/repo(.git)
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    # https://host/owner/repo(.git)
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0

def url_host_and_mode(url: str) -> Tuple[Optional[str], str]:
    """
    Returns (host, mode) where mode is:
      - "ssh" for git@host:owner/repo
      - "https" for https://host/...
      - "other" otherwise
    """
    u = url.strip()
    if re.match(r"^[^@]+@([^:]+):", u):
        host = u.split("@", 1)[1].split(":", 1)[0]
        return host, "ssh"
    if "://" in u:
        parsed = urlparse(u)
        host = (parsed.netloc or "").split("@")[-1].split(":")[0] or None
        return host, parsed.scheme.lower()
    return None, "other"

def git_auth_config_args(host: str, token: str) -> List[str]:
    """
    Injects an HTTP Authorization header for git HTTPS requests to the given host.
    Uses Basic auth with x-access-token:<TOKEN>.
    """
    basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
    # -c http.https://HOST/.extraheader="Authorization: Basic ...."
    return ["-c", f"http.https://{host}/.extraheader=Authorization: Basic {basic}"]

def sh(
    cmd: List[str],
    cwd: Optional[Path] = None,
    check: bool = True,
    capture: bool = True,
    timeout: Optional[int] = DEFAULT_TIMEOUT,
    auth_url: Optional[str] = None,
    token: Optional[str] = None
) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)

    if cmd and cmd[0] == "git" and token and auth_url:
        host, mode = url_host_and_mode(auth_url)
        # Only apply token injection for HTTPS-like URLs (tokens don't apply to SSH)
        if host and mode in ("https", "http"):
            cmd = ["git", *git_auth_config_args(host, token), *cmd[1:]]

    return subprocess.run(
        cmd, cwd=cwd, check=check,
        capture_output=capture, text=True,
        timeout=timeout, env=env
    )

class TokenRotator:
    def __init__(self, tokens: List[str]) -> None:
        self.tokens = tokens[:]
        self.i = 0

    def next(self) -> Tuple[int, str]:
        """
        Returns (slot, token) where slot is 1..N.
        """
        if not self.tokens:
            raise RuntimeError("No tokens available")
        idx = self.i % len(self.tokens)
        self.i += 1
        return (idx + 1, self.tokens[idx])

def is_retryable_auth_error(msg: str) -> bool:
    m = (msg or "").lower()
    # Covers common GitHub/HTTP auth & throttling patterns seen in git stderr
    needles = [
        "rate limit", "abuse detection", "too many requests", "http 429",
        "http 403", "403 forbidden", "http 401", "401 unauthorized",
        "authentication failed", "could not read username", "access denied",
        "fatal: unable to access", "remote: permission to"
    ]
    return any(n in m for n in needles)

def ensure_full_clone_no_submodules_no_lfs(
    url: str,
    dest_root: Path,
    fetch_pr_refs: bool,
    token: Optional[str] = None
) -> Path:
    """
    Ensures a non-shallow clone with full history of all branches & tags.
    NO submodules. NO LFS.
    Optionally fetches GitHub PR heads.
    """
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if not (d.exists() and (d / ".git").exists()):
        # Full clone (not shallow). Explicitly no submodules.
        sh(
            ["git", "clone", "--no-single-branch", "--no-recurse-submodules", "--tags", "--quiet", url, str(d)],
            capture=True,
            auth_url=url,
            token=token
        )
    else:
        # Ensure origin URL is correct
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False, capture=True, auth_url=url, token=token)

    # Helpful on Windows for deep trees (best-effort)
    sh(["git", "config", "core.longpaths", "true"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    # If shallow, unshallow
    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False, capture=True, auth_url=url, token=token)
    if cp.returncode == 0 and (cp.stdout or "").strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags", "--quiet"], cwd=d, capture=True, auth_url=url, token=token)
    else:
        sh(["git", "fetch", "--tags", "--quiet"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    # Fetch all branches under refs/heads/* and prune deleted ones
    sh(
        ["git", "fetch", "origin", "--prune", "--tags",
         "+refs/heads/*:refs/remotes/origin/*", "--quiet"],
        cwd=d, check=False, capture=True, auth_url=url, token=token
    )

    # Best-effort PR refs (GitHub); harmless if not present
    if fetch_pr_refs:
        sh(
            ["git", "fetch", "origin",
             "+refs/pull/*/head:refs/remotes/origin/pr/*", "--quiet"],
            cwd=d, check=False, capture=True, auth_url=url, token=token
        )

    return d

def clone_with_token_rotation(
    url: str,
    clone_root: Path,
    fetch_pr_refs: bool,
    rotator: Optional[TokenRotator],
    max_token_attempts: int = 6
) -> Tuple[Path, Dict[str, object]]:
    """
    Tries clone/fetch using rotating tokens (HTTPS only). For SSH URLs, runs once (no token).
    Returns (repo_path, meta)
    """
    host, mode = url_host_and_mode(url)
    meta: Dict[str, object] = {
        "auth_mode": None,
        "token_slot": None,
        "attempts": 0,
    }

    # SSH or non-https -> no token injection possible/needed
    if mode == "ssh":
        meta["auth_mode"] = "ssh"
        meta["attempts"] = 1
        d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
        return d, meta

    # HTTPS with token rotation if we have tokens
    if rotator and mode in ("https", "http") and host:
        meta["auth_mode"] = "https-token"
        last_err: Optional[Exception] = None

        # Try up to N tokens (or fewer if you configured fewer)
        tries = min(max_token_attempts, len(rotator.tokens))
        for _ in range(tries):
            slot, token = rotator.next()
            meta["attempts"] += 1
            meta["token_slot"] = slot

            try:
                d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=token)
                return d, meta
            except subprocess.CalledProcessError as e:
                # Retry only if it looks like auth/rate/throttle; otherwise fail fast.
                err = (e.stderr or e.stdout or str(e))[:5000]
                last_err = e
                if is_retryable_auth_error(err):
                    log(f"Retryable auth/rate error for {url} using token slot {slot}; rotating token...")
                    continue
                raise

        # exhausted tokens
        if last_err:
            raise last_err
        raise RuntimeError("Clone failed after token rotation attempts")

    # HTTPS but no tokens loaded
    meta["auth_mode"] = "https-no-token"
    meta["attempts"] = 1
    d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
    return d, meta


# -----------------------------
# Main
# -----------------------------
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

# Load tokens
rotator: Optional[TokenRotator] = None
try:
    load_env_file(TOKENS_ENV_FILE)
    tokens = [os.environ.get(k) for k in TOKEN_KEYS]
    tokens = [t for t in tokens if t and t.strip()]
    if tokens:
        rotator = TokenRotator(tokens)
        log(f"Loaded {len(tokens)} GitHub token(s) from {TOKENS_ENV_FILE.name} (slots: 1..{len(tokens)})")
    else:
        log(f"No tokens found in {TOKENS_ENV_FILE.name}; proceeding without tokens.")
except Exception as e:
    log(f"Token file not loaded ({e}); proceeding without tokens.")

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue

        t0 = time.time()
        rec: Dict[str, object] = {
            "repo_url": url,
            "dir": None,
            "status": "unknown",
            "seconds": None,
            "total_commits": None,
            "error": "",

            # token/logging fields
            "auth_mode": None,
            "token_slot": None,
            "attempts": None,
        }

        try:
            d, meta = clone_with_token_rotation(
                url=url,
                clone_root=CLONE_ROOT,
                fetch_pr_refs=FETCH_PR_REFS,
                rotator=rotator,
                max_token_attempts=6
            )
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            rec.update(meta)
            ok += 1

            log(f"[ok] {url} -> {rec['dir']}  commits={rec['total_commits']}  "
                f"auth={rec['auth_mode']} token_slot={rec['token_slot']} attempts={rec['attempts']}")

        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
            log(f"[error] {url}  auth={rec.get('auth_mode')} token_slot={rec.get('token_slot')} attempts={rec.get('attempts')}")

        except Exception as e:
            rec["status"] = "error"
            rec["error"]  = str(e)[:2000]
            fail += 1
            log(f"[error] {url}  ({e})")

        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)

# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = [
    "repo_url","dir","status","seconds","total_commits","error",
    "auth_mode","token_slot","attempts"
]
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

log(f"Done. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[2025-12-14 22:05:26] Loaded 6 GitHub token(s) from All_Tokens.env (slots: 1..6)
[2025-12-14 22:05:28] [ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\connectbot__connectbot  commits=4100  auth=https-token token_slot=1 attempts=1
[2025-12-14 22:05:29] [ok] https://github.com/ge0rg/aprsdroid -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\ge0rg__aprsdroid  commits=1211  auth=https-token token_slot=2 attempts=1
[2025-12-14 22:06:07] [ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\robolectric__robolectric  commits=21638  auth=https-token token_slot=3 attempts=1
[2025-12-14 22:06:36] [ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone\opendocument-app__OpenDocument.droid  c

## Step 2: Mining 
add a baseline to read the repos from the begining not from the first change

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor as PoolExecutor, as_completed
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple

# ===========================
# Config (EDIT THESE PATHS)
# ===========================
WORK_ROOT    = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2")
INPUT_CLONES = WORK_ROOT / "Clone"     
OUTPUT_MINE  = WORK_ROOT / "Mine"

# Study end date (UTC)
CUTOFF_ISO = "2025-08-10 23:59:59 +0000"

# IMPORTANT: Default branch ONLY (first-parent)
TIMELINE_SCOPE = "default_first_parent"  # fixed; do not change

# Performance / behavior toggles
MAX_REPOS = 0  # 0 = all
MAX_WORKERS = min(32, (os.cpu_count() or 8) * 2)
RESUME_IF_EXISTS = True
SUPPRESS_EMPTY_ROWS = True
CAPTURE_EMPTY_GAPS = True
EMIT_NONE_STATE = False

# Blob size guards
BLOB_MAX_SIZE = 2_000_000
HARD_MAX_SIZE = 5_000_000

# Workflow-expansion bounds (prevents explosion)
EXPAND_MAX_DEPTH = 3
EXPAND_MAX_FILES = 250
EXPAND_MAX_TEXT_CHARS = 5_000_000

# High-priority CI paths never skipped (even if large)
HIGH_PRIORITY_CI_PATHS = {
    ".github/workflows",
    ".gitlab-ci.yml",
    "azure-pipelines.yml",
    ".circleci/config.yml",
    ".bitrise.yml",
    ".travis.yml",
}

# CI entrypoints (cut-off procedure)
CI_ENTRYPOINTS_EXACT = {
    ".travis.yml",
    ".gitlab-ci.yml",
    "azure-pipelines.yml",
    "circle.yml",
    ".bitrise.yml",
    ".circleci/config.yml",
}
GHA_WORKFLOWS_DIR = ".github/workflows"

# ===========================
# Quick startup checks
# ===========================
def _failfast_checks() -> None:
    if shutil.which("git") is None:
        print("[fatal] Git not found in PATH. Install Git and/or add it to PATH.", file=sys.stderr)
        raise SystemExit(1)
    if not WORK_ROOT.exists():
        print(f"[fatal] WORK_ROOT does not exist: {WORK_ROOT}", file=sys.stderr)
        raise SystemExit(1)
    if not INPUT_CLONES.exists():
        print(f"[warn] INPUT_CLONES does not exist yet: {INPUT_CLONES}")
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

# ===========================
# File classifiers
# ===========================
YAML_EXTS = (".yml", ".yaml")
GRADLE_NAMES = {"build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts"}
GRADLE_EXTS = (".gradle", ".gradle.kts")

SCRIPT_EXTS = (".sh", ".bat", ".cmd", ".ps1", ".py", ".js", ".rb", ".pl")
CI_BUILD_SPECIAL = {"Jenkinsfile", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml", ".bitrise.yml"}
XML_BUILD_FILES = {"pom.xml", "build.xml", "config.xml"}

def is_yaml(path: str) -> bool:
    return os.path.splitext(path)[1].lower() in YAML_EXTS or os.path.basename(path) in CI_BUILD_SPECIAL

def is_gradle(path: str) -> bool:
    name = os.path.basename(path)
    ext = os.path.splitext(path)[1].lower()
    return name in GRADLE_NAMES or ext in GRADLE_EXTS

def is_script(path: str) -> bool:
    ext = os.path.splitext(path)[1].lower()
    return ext in SCRIPT_EXTS or os.path.basename(path) in ("Jenkinsfile",)

def is_ci_xml_or_build_xml(path: str) -> bool:
    return os.path.basename(path).lower() in {n.lower() for n in XML_BUILD_FILES}

def is_relevant_file(path: str) -> bool:
    return is_yaml(path) or is_gradle(path) or is_script(path) or is_ci_xml_or_build_xml(path)

def is_high_priority_ci_path(path: str) -> bool:
    norm = path.replace("\\", "/")
    base = os.path.basename(norm)
    if base in HIGH_PRIORITY_CI_PATHS:
        return True
    for root in HIGH_PRIORITY_CI_PATHS:
        if norm.startswith(root.rstrip("/") + "/"):
            return True
    return False

# ===========================
# Regex helpers & normalizers
# ===========================
COMMENT_LINE_RE = re.compile(r"(?m)^\s*(#|//|REM\b|::).*?$")

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# --- Gradle comment stripper (preserves http(s)://) ---
def strip_comments_gradle(text: str) -> str:
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.S)                 # /* ... */
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.M)                  # // ...
    return s

def normalize_block_keys(text: str) -> str:
    # expose content of run/script/command keys and drop YAML dashes before likely shell lines
    text = re.sub(r"(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$", r"\2", text)
    text = re.sub(r"(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$", "", text)
    text = re.sub(
        r"(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|python(?:3)?|node|ruby|perl|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)",
        r"\1",
        text,
    )
    return text

IGNORE_GHA_ACTIONS_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*(docker/(?:setup-qemu-action|setup-buildx-action|build-push-action|login-action)|actions/checkout|docker/setup-qemu-action|docker/setup-buildx-action)@.*$"
)

def strip_irrelevant_ci_lines(text: str) -> str:
    return IGNORE_GHA_ACTIONS_RE.sub("", text or "")

# --- Excluded Gradle task sanitizer (align with YAML scanner) ---
EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)

def remove_excluded_gradle_tasks(text: str) -> str:
    return EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")

GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def pre_sanitize_for_triggers(text: str) -> str:
    # For detection matching: remove excluded tasks + GHA expressions
    t = remove_excluded_gradle_tasks(text or "")
    return GHA_EXPR_RE.sub("", t)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def compile_any(patterns: List[str], flags: int = re.I | re.M) -> List[re.Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable, text: str) -> bool:
    for p in patterns:
        if isinstance(p, str):
            p = re.compile(p, re.I | re.M)
        if p.search(text):
            return True
    return False

# ===========================
# Detection patterns (core)
# ===========================
GRADLE_PREFIX = (
    r"^\s*"
    r"(?:\S+=\S+\s+)*"
    r"(?:sudo\s+)?"
    r"(?:(?:bash|sh)\s+-c[l]?\s+[\'\"]?)?"
    r"(?:[^#\n;]*?&&\s+)?"
    r"(?:cd\s+\S+\s+&&\s+)?"
    r"(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?"
)
GRADLE_ANYWHERE = r"(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*"
GRADLE_ANYWHERE_RE = re.compile(GRADLE_ANYWHERE)
GRADLE_BUILD_ACTION_RE = re.compile(r"(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@")
NON_TEST_PREFIX = r"(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)"
SHELL_PREFIX = r"(?:\S+=\S+\s+)*(?:sudo\s+)?(?:(?:bash|sh|pwsh|powershell|python(?:3)?|node|ruby|perl)\s+-c\s+[\'\"]?)?(?:[^#\n;]*?&&\s+)?"

EMULATOR_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}(?:\s|^)(?:(?:\./|\.\\)?(?:emulator)(?:\.exe)?)\b[^\n]*-avd\s+\S+"
ADB_WAIT_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b"
ADB_SERIAL_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)"

# NOTE: Real devices are out of scope for evolution. We keep patterns out of styles/events.
# If you truly want to “not detect” them at all, remove the Real_Device entry below.
REAL_DEVICE_LINE = rf"(?mi)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b"

DEVICE_SOURCES = [
    # (Optional, out-of-scope for style evolution; kept only for strict Flutter gating parity)
    ("Real_Device", "adb -s <serial> (physical)", [REAL_DEVICE_LINE]),

    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device", [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@", [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r"(?mi)^\s*(?:\./)?android-wait-for-emulator\b"]),
    ("Emulator", "start-emulator.sh", [r"(?mi)^\s*start-emulator\.sh\b"]),
    ("Emulator", "android create avd", [r"\bandroid\b[^\n]*\bcreate\s+avd\b"]),
    ("Emulator", "circleci android orb", [
        r"(?mi)^\s*(?:-\s*)?android/(?:start-emulator-and-run-tests|create-avd|launch-emulator)\s*:",
        r"(?mi)^\s*system-image\s*:\s*system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*);(?:default|google_apis)[^\s]*"
    ]),
    ("Emulator", "reactivecircus runner", [r"(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+"]),
    ("Emulator", "malinskiy runner", [r"(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+"]),
    ("Emulator", "sys-img component", [
        r"(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b",
        r"(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b"
    ]),
    ("Emulator", "avdmanager", [r"(?m)^\s*\S*avdmanager\b"]),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r"^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b"
    ]),
    ("Emulator", "other gha emulator", [
        r"(?mi)^\s*uses\s*:\s*vgaidarji/android-github-actions-emulator@[\w\.\-]+",
        r"(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)(?!malinskiy/action-android/emulator-run-cmd@)(?!emulator-wtf/run-tests@)[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@[\w\.\-]+"
    ]),
    ("Third_Party_Lab", "gcloud firebase", [r"(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b"]),
    ("Third_Party_Lab", "flank", [r"(?mi)^[^\n]*\bflank\s+android\s+run\b"]),
    ("Third_Party_Lab", "saucectl", [r"(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b"]),
    ("Third_Party_Lab", "browserstack/bstack", [r"(?i)\b(browserstack|bstack)\b"]),
    ("Third_Party_Lab", "appcenter test", [r"(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b"]),
    ("Third_Party_Lab", "maestro cloud", [r"(?mi)^[^\n]*\bmaestro\s+cloud\b"]),
    ("Third_Party_Lab", "emulator.wtf action", [
        r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+",
        r"(?i)\bemulator\.wtf\b"
    ]),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

TRIGGER_SOURCES_PRIMARY = [
    ("Gradle", "connectedAndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*"]),
    ("Gradle", "connected.*Android.*", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b(?![^\n\r]*\b(?:{NON_TEST_PREFIX})[\w-]*androidtest\b)[^\n\r]*"]),
    ("Gradle", "connectedCheck", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*"]),
    ("Gradle", "cAT shorthand", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*"]),
    ("Gradle", "deviceCheck", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*"]),
    ("Gradle", "managedDevice AndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*"]),
    ("Gradle", "variant/device AndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b[^\n\r]*"]),
    ("Gradle", "Spoon", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b"]),
    ("Gradle", "Marathon", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b"]),
    ("ADB", "am instrument", [r"(?mi)^[^\n]*\bam\s+instrument\b"]),
    ("Third_Party_Lab", "gcloud firebase (instr)", [r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)"]),
    ("Third_Party_Lab", "flank", [r"(?mi)^[^\n]*\bflank\s+android\s+run\b"]),
    ("Third_Party_Lab", "saucectl", [r"(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b"]),
    ("Third_Party_Lab", "appcenter", [r"(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b"]),
    ("Third_Party_Lab", "emulator.wtf run", [r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+", r"(?i)\bemulator\.wtf\b"]),
    # BaselineProfile & benchmark tasks (kept as in your miner)
    ("Gradle", "generateBaselineProfile", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generate(?:\w*?)baselineprofile\b[^\n\r]*"]),
    ("Gradle", "collectBaselineProfile", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collect(?:\w*?)baselineprofile\b[^\n\r]*"]),
    ("Gradle", "connectedBenchmarkAndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*"]),
]
TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]

TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b"]),
    ("Gradle", "connectedAndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bconnectedandroidtest\b"]),
    ("Gradle", "connectedCheck (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connectedcheck\b"]),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b"]),
    ("Gradle", "variant/device AndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})[\w-]*androidtest\b"]),
    ("Gradle", "Spoon (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b"]),
    ("Gradle", "Marathon (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b"]),
    ("Gradle", "generateBaselineProfile (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bgenerate(?:\w*?)baselineprofile\b"]),
    ("Gradle", "collectBaselineProfile (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bcollect(?:\w*?)baselineprofile\b"]),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bconnectedbenchmarkandroidtest\b"]),
]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

def collect_hits_with_groups(patterns, text: str):
    labels = []
    groups = []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# --- GHA Gradle inputs (kept) ---
GHA_GRADLE_INPUTS = compile_any([
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:(?:[:\w-]+:)*) (?!assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)[\w-]*androidtest\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b",
])

# --- NEW: GHA GMD inputs (align with YAML scanner) ---
GHA_GMD_INPUTS = compile_any([
    rf"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?!(?:[:\w-]+:)*(?:connected[a-z0-9:._-]*|{NON_TEST_PREFIX})[\w:-]*androidtest\b)(?:[:\w-]+:)*[\w:-]*androidtest\b"
])

# Provider patterns (unchanged)
PROVIDER_PATTERNS = [
    ("emulator-wtf", compile_any([r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@", r"(?i)\bemulator\.wtf\b"])),
    ("firebase-test-lab", compile_any([r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b", r"(?mi)\bflank\s+android\s+run\b"])),
    ("browserstack", compile_any([r"(?i)\bbrowserstack\b", r"(?i)\bbstack\b"])),
    ("sauce-labs", compile_any([r"(?mi)\bsaucectl(?:\s+run)?\b", r"(?mi)\bsauce\s+ctl\b"])),
    ("appcenter", compile_any([r"(?mi)\bappcenter\s+test\s+run\s+android\b"])),
    ("maestro-cloud", compile_any([r"(?mi)\bmaestro\s+cloud\b"])),
]

INLINE_DEVICE_HINTS = compile_any([
    r"(?mi)^\s*devices\s*:\s*\|",
    r"(?mi)\b--device\b",
    r"(?mi)\bmodel\s*=\s*[^,\s]+",
    r"(?mi)\bversion\s*=\s*\d+",
    r"(?mi)\blocale\s*=\s*[-\w]+",
    r"(?mi)\borientation\s*=\s*(portrait|landscape)",
    r"(?mi)^\s*with-orchestrator\s*:\s*true\b",
    r"(?mi)\b--use-orchestrator\b",
    r"(?mi)\bnum-flaky-test-attempts\s*:\s*\d+\b",
    r"(?mi)\b--num-flaky-test-attempts(?:=|\s+)\d+\b",
])

CONFIG_FILE_HINTS = compile_any([
    r"(?mi)\.ewtf\.ya?ml\b",
    r"(?mi)\b(flank\.ya?ml|flank\.android\.ya?ml)\b",
    r"(?mi)\b--config(?:=|\s+)\S+",
    r"(?mi)\b(browserstack\.ya?ml)\b",
    r"(?mi)\b(bs(?:config)?\.ya?ml)\b",
])

ANDROID_CONTEXT_RE = re.compile(
    r"(?i)\b(adb|avd|emulator|android\s+sdk|system-images;android-|androidtest|connected(?:check|androidtest)|gcloud\s+firebase\s+test\s+android\s+run)\b"
)

# ===========================
# NEW: STRICT Flutter gating (align with YAML scanner)
# ===========================
FLUTTER_IT_LINE = re.compile(r"(?mi)^\s*flutter\s+(?:test|drive)\b[^\n]*")
FLUTTER_IT_ANDROID_HINT = re.compile(r"(?i)(integration_test|--driver\b|/integration_test/)")
FLUTTER_DEVICE_FLAG_RE = re.compile(r"(?i)\s+-d\s+(?P<dev>\"[^\"]+\"|'[^']+'|\S+)")
FLUTTER_DEVICE_IS_ANDROID = re.compile(r"(?i)\b(android|emulator-\d+|sdk\s+gphone|android\s+sdk\s+built\s+for|pixel)\b")

ANDROID_RUNTIME_ENV_LABELS = {
    "emulator -avd/@",
    "adb wait-for-device",
    "adb -s emulator-serial",
    "android-wait-for-emulator",
    "start-emulator.sh",
    "reactivecircus runner",
    "malinskiy runner",
    "other gha emulator",
    "circleci android orb",
    # real device label (optional)
    "adb -s <serial> (physical)",
}
ANDROID_STRICT_3P_LABELS = {
    "gcloud firebase",
    "appcenter test",
    "emulator.wtf action",
}

def has_android_runtime_evidence(dev_labels: List[str], dev_groups: List[str]) -> bool:
    lbls = {str(l).strip().lower() for l in (dev_labels or []) if str(l).strip()}
    grps = {str(g).strip() for g in (dev_groups or []) if str(g).strip()}

    if "Real_Device" in grps:
        return True
    if lbls & {s.lower() for s in ANDROID_RUNTIME_ENV_LABELS}:
        return True
    if lbls & {s.lower() for s in ANDROID_STRICT_3P_LABELS}:
        return True
    return False

def flutter_androidish_in_block(block_text: str, dev_labels: List[str], dev_groups: List[str]) -> bool:
    """
    STRICT: Flutter integration tests count only if they explicitly target Android (-d android / emulator-* etc.)
    OR the same block has strong Android runtime evidence (emulator started/waited, emulator runners, etc.).
    """
    hits: List[str] = []
    for m in FLUTTER_IT_LINE.finditer(block_text or ""):
        line = m.group(0) or ""
        if FLUTTER_IT_ANDROID_HINT.search(line) or FLUTTER_DEVICE_IS_ANDROID.search(line):
            hits.append(line)

    if not hits:
        return False

    android_targeted = False
    for line in hits:
        d = FLUTTER_DEVICE_FLAG_RE.search(line)
        if d:
            plat = (d.group("dev") or "").strip('"\''"").lower()
            if (
                plat == "android"
                or plat.startswith("emulator-")
                or "sdk gphone" in plat
                or "android sdk built for" in plat
                or "pixel" in plat
            ):
                android_targeted = True
                break

    if android_targeted:
        return True

    return has_android_runtime_evidence(dev_labels, dev_groups)

# ===========================
# Environment label sets (your updated mapping: DIY->Custom, Malinskiy/Other/ReactiveCircus->Community)
# ===========================
EMULATOR_COMMUNITY_LABELS = {"reactivecircus runner", "malinskiy runner", "other gha emulator", "circleci android orb"}
EMULATOR_CUSTOM_LABELS = {
    "adb -s emulator-serial",
    "adb wait-for-device",
    "emulator -avd/@",
    "android-wait-for-emulator",
    "start-emulator.sh",
    "android create avd",
    "avdmanager",
    "sdkmanager system-images/emulator",
    "sys-img component",
}
THIRD_PARTY_STRONG_LABELS = {"gcloud firebase", "emulator.wtf run", "saucectl", "appcenter test", "browserstack/bstack", "maestro cloud"}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
STRONG_DEVICE_LABELS = EMULATOR_COMMUNITY_LABELS | EMULATOR_CUSTOM_LABELS | THIRD_PARTY_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS

def detect_provider(text: str) -> str:
    for name, pats in PROVIDER_PATTERNS:
        if any_match(pats, text):
            return name
    return "other"

def has_inline_env(text: str) -> bool:
    return any_match(INLINE_DEVICE_HINTS, text)

def has_config_env(text: str) -> bool:
    return any_match(CONFIG_FILE_HINTS, text)

# ===========================
# Gradle GMD config detector
# ===========================
TESTOPTIONS_HEAD = re.compile(r"\btestOptions\s*\{", re.I)
MANAGED_HEAD     = re.compile(r"\bmanagedDevices\s*\{", re.I)
GMD_INNER_TOKENS_RX = re.compile(r"\b(managedDevices|managedVirtualDevice|devices|groups|deviceGroups)\b", re.I)

def _find_block_span_from_head(text: str, head_start: int) -> Optional[Tuple[int, int]]:
    i = text.find("{", head_start)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return (i, j)
    return None

def find_gmd_block(text_gradle_no_comments: str) -> bool:
    t = text_gradle_no_comments
    for m in MANAGED_HEAD.finditer(t):
        if _find_block_span_from_head(t, m.start()):
            return True
    for m in TESTOPTIONS_HEAD.finditer(t):
        ci = _find_block_span_from_head(t, m.start())
        if not ci:
            continue
        a, b = ci
        sub = t[a:b+1]
        if GMD_INNER_TOKENS_RX.search(sub):
            return True
    return False

# ===========================
# Git helpers
# ===========================
def run_git(repo: Path, args: List[str], text: bool = True, check: bool = True, retries: int = 2) -> subprocess.CompletedProcess:
    last_exc: Optional[BaseException] = None
    for attempt in range(retries + 1):
        try:
            return subprocess.run(
                ["git", "-C", str(repo)] + args,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=text,
                check=check,
                encoding="utf-8" if text else None,
                errors="replace" if text else None,
            )
        except subprocess.CalledProcessError as e:
            last_exc = e
            time.sleep(0.1 * (attempt + 1) + random.random() * 0.1)
        except Exception as e:
            last_exc = e
            time.sleep(0.05)
    if isinstance(last_exc, subprocess.CalledProcessError) and not check:
        return last_exc  # type: ignore[return-value]
    raise last_exc  # type: ignore[misc]

def list_repos(root: Path) -> List[Path]:
    out: List[Path] = []
    for p, dirs, _files in os.walk(root):
        pth = Path(p)
        if (pth / ".git").exists():
            out.append(pth)
            dirs[:] = []
    return out

def default_branch_ref(repo: Path) -> Optional[str]:
    try:
        ref = run_git(repo, ["symbolic-ref", "refs/remotes/origin/HEAD"], check=False).stdout.strip()
        if ref.startswith("refs/remotes/origin/"):
            return "origin/" + ref.split("refs/remotes/origin/")[1]
    except Exception:
        pass
    for cand in ("origin/main", "origin/master"):
        cp = run_git(repo, ["rev-parse", "--verify", cand], check=False)
        if cp.returncode == 0:
            return cand
    return None

def cutoff_head_default(repo: Path, cutoff_iso: str) -> Optional[str]:
    ref = default_branch_ref(repo)
    if not ref:
        return None
    cp = run_git(repo, ["rev-list", "-n1", "--first-parent", f"--before={cutoff_iso}", ref], check=False)
    sha = (cp.stdout or "").strip()
    return sha or None

def earliest_commit_first_parent(repo: Path, head: str) -> Optional[str]:
    cp = run_git(repo, ["rev-list", "--max-parents=0", "--first-parent", head], check=False)
    s = (cp.stdout or "").strip()
    return s.splitlines()[0] if s else None

def get_commit_date_iso(repo: Path, commit: str) -> Optional[str]:
    try:
        return run_git(repo, ["show", "-s", "--format=%cI", commit]).stdout.strip() or None
    except Exception:
        return None

def commits_touching_relevant_default(repo: Path, head: str) -> List[str]:
    pathspecs = [
        f":(glob){GHA_WORKFLOWS_DIR}/**/*.yml",
        f":(glob){GHA_WORKFLOWS_DIR}/**/*.yaml",
        ".travis.yml",
        ".gitlab-ci.yml",
        "azure-pipelines.yml",
        ".circleci/config.yml",
        ".bitrise.yml",
        "circle.yml",
        ":(glob)**/*.yml",
        ":(glob)**/*.yaml",
        "build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts",
        ":(glob)**/*.gradle", ":(glob)**/*.gradle.kts",
        "Jenkinsfile",
        ":(glob)**/*.sh", ":(glob)**/*.bat", ":(glob)**/*.cmd", ":(glob)**/*.ps1",
        ":(glob)**/*.py", ":(glob)**/*.js", ":(glob)**/*.rb", ":(glob)**/*.pl",
        ":(glob)**/pom.xml", ":(glob)**/build.xml", ":(glob)**/config.xml",
    ]
    s = run_git(repo, ["rev-list", "--reverse", "--first-parent", head, "--"] + pathspecs, check=False).stdout
    return [c for c in s.splitlines() if c.strip()]

def list_tree_entries(repo: Path, treeish: str) -> List[Tuple[str, str]]:
    raw = run_git(repo, ["ls-tree", "-r", "-z", treeish], check=False).stdout or ""
    out: List[Tuple[str, str]] = []
    for entry in raw.split("\x00"):
        if not entry or "\t" not in entry:
            continue
        meta, path = entry.split("\t", 1)
        parts = meta.split()
        if len(parts) < 3:
            continue
        sha = parts[2]
        out.append((sha, path.replace("\\", "/")))
    return out

def blob_size(repo: Path, sha: str) -> int:
    try:
        return int(run_git(repo, ["cat-file", "-s", sha], check=False).stdout.strip())
    except Exception:
        return 0

@lru_cache(maxsize=350_000)
def read_blob_cached(repo_path: str, sha: str) -> Optional[str]:
    repo = Path(repo_path)
    try:
        return run_git(repo, ["cat-file", "-p", sha], check=False).stdout
    except Exception:
        return None

# ===========================
# Workflow expansion (resolve referenced scripts/actions)
# ===========================
RX_USES = re.compile(r"(?mi)^\s*uses\s*:\s*([^\s#]+)\s*$")
RX_CD_PREFIX = re.compile(r"(?i)^\s*cd\s+([^\s;&|]+)\s*(?:&&|\;)\s*(.+)$")
RX_WORKDIR = re.compile(r"(?mi)^\s*working-directory\s*:\s*([^\n#]+)$")

RX_EXEC_REL = re.compile(r"""(?x)(?:^|[\s;&|()])((?:\./|\.\\)[^\s"'`<>]+)""")
RX_LANG_CALL = re.compile(r"""(?ix)(?:^|[\s;&|()])(bash|sh|pwsh|powershell|python3?|node|ruby|perl)\s+([^\s"'`<>]+)""")
RX_CONFIG_ARG = re.compile(r"""(?ix)\b--config(?:=|\s+)([^\s"'`<>]+)""")
RX_DYNAMIC_MARKERS = re.compile(r"(\${{\s*[^}]+}}|\$[A-Za-z_][A-Za-z0-9_]*|%[A-Za-z_][A-Za-z0-9_]*%)")

def _clean_ref_token(tok: str) -> str:
    t = tok.strip()
    t = t.strip('\'"')
    t = t.rstrip("),;")
    return t

def _is_probably_dynamic(path: str) -> bool:
    return bool(RX_DYNAMIC_MARKERS.search(path))

def _normalize_rel_path(p: str) -> str:
    p = p.replace("\\", "/")
    if p.startswith("./"):
        p = p[2:]
    p = str(Path(p).as_posix())
    if p.startswith("./"):
        p = p[2:]
    return p

def _join_posix(a: str, b: str) -> str:
    return str((Path(a) / b).as_posix()).replace("\\", "/")

def _candidate_paths_for_ref(ref: str, current_file: str) -> List[str]:
    r = _normalize_rel_path(ref)
    cur_dir = str(Path(current_file).parent.as_posix()).replace("\\", "/")
    cand = []
    if r:
        cand.append(r)
        if cur_dir and cur_dir != ".":
            cand.append(_join_posix(cur_dir, r))
    out = []
    seen = set()
    for x in cand:
        x = x.lstrip("/")
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _resolve_local_action_metadata(dir_path: str) -> List[str]:
    d = dir_path.rstrip("/").lstrip("/")
    return [f"{d}/action.yml", f"{d}/action.yaml"]

def extract_references_from_text(text: str, current_file: str) -> Tuple[Set[str], Set[str], int, List[str]]:
    """
    Returns:
      - local_action_dirs: directories referenced by uses: ./...
      - file_refs: file paths invoked (scripts/configs) relative-ish
      - unresolved_dynamic_count
      - unresolved_samples
    """
    local_action_dirs: Set[str] = set()
    file_refs: Set[str] = set()
    unresolved = 0
    unresolved_samples: List[str] = []

    if not text:
        return local_action_dirs, file_refs, unresolved, unresolved_samples

    t = strip_comments(text)
    t = normalize_block_keys(t)
    t = strip_irrelevant_ci_lines(t)

    # NEW: capture working-directory hints (align with YAML scanner)
    base_workdirs: List[str] = []
    for m in RX_WORKDIR.finditer(t):
        wd = _clean_ref_token(m.group(1) or "")
        if wd:
            wd_norm = _normalize_rel_path(wd).rstrip("/")
            if wd_norm and wd_norm not in base_workdirs:
                base_workdirs.append(wd_norm)

    # Keep expressions for unresolved counting; only remove excluded-task segments here.
    # (Do NOT strip ${{ }} here, otherwise dynamic refs won’t be counted.)
    t_for_refs = remove_excluded_gradle_tasks(t)

    # uses: lines
    for m in RX_USES.finditer(t_for_refs):
        u = (m.group(1) or "").strip()
        if not u:
            continue
        if u.startswith("./") or u.startswith(".\\"):
            u2 = _normalize_rel_path(u)
            local_action_dirs.add(u2.rstrip("/"))

    for line in t_for_refs.splitlines():
        line = line.strip()
        if not line:
            continue

        # working dir candidates = repo root + YAML working-directory + any "cd X && ..."
        work_dirs = [""] + base_workdirs

        cur = line
        for _ in range(2):
            mcd = RX_CD_PREFIX.match(cur)
            if not mcd:
                break
            wd = _clean_ref_token(mcd.group(1) or "")
            rest = mcd.group(2) or ""
            if wd:
                wd_norm = _normalize_rel_path(wd).rstrip("/")
                if wd_norm and wd_norm not in work_dirs:
                    work_dirs.append(wd_norm)
            cur = rest.strip()

        # ./relative_exec
        for mm in RX_EXEC_REL.finditer(cur):
            p = _clean_ref_token(mm.group(1) or "")
            if not p:
                continue
            if _is_probably_dynamic(p):
                unresolved += 1
                if len(unresolved_samples) < 10:
                    unresolved_samples.append(p)
                continue
            p2 = _normalize_rel_path(p)
            for wd in work_dirs:
                if wd:
                    file_refs.add(_join_posix(wd, p2))
                else:
                    file_refs.add(p2)

        # bash/python/node <path>
        for mm in RX_LANG_CALL.finditer(cur):
            p = _clean_ref_token(mm.group(2) or "")
            if not p:
                continue
            if p.startswith("http://") or p.startswith("https://"):
                continue
            if _is_probably_dynamic(p):
                unresolved += 1
                if len(unresolved_samples) < 10:
                    unresolved_samples.append(p)
                continue
            p2 = p.replace("\\", "/").lstrip("./")
            for wd in work_dirs:
                if wd:
                    file_refs.add(_join_posix(wd, p2))
                else:
                    file_refs.add(p2)

        # --config=FILE
        for mm in RX_CONFIG_ARG.finditer(cur):
            p = _clean_ref_token(mm.group(1) or "")
            if not p:
                continue
            if _is_probably_dynamic(p):
                unresolved += 1
                if len(unresolved_samples) < 10:
                    unresolved_samples.append(p)
                continue
            p2 = p.replace("\\", "/").lstrip("./")
            file_refs.add(p2)

    return local_action_dirs, file_refs, unresolved, unresolved_samples

# ===========================
# Snapshot scan (DEFAULT branch commit) with workflow expansion
# ===========================
def scan_treeish_for_styles_expanded(repo: Path, treeish: str) -> Tuple[Set[str], Dict]:
    entries = list_tree_entries(repo, treeish)
    path_to_sha: Dict[str, str] = {p: sha for (sha, p) in entries}

    def read_path(p: str) -> Optional[str]:
        sha = path_to_sha.get(p)
        if not sha:
            return None
        size = blob_size(repo, sha)
        if not is_high_priority_ci_path(p) and BLOB_MAX_SIZE and size > BLOB_MAX_SIZE:
            return None
        txt = read_blob_cached(str(repo), sha)
        if not txt or len(txt) > HARD_MAX_SIZE:
            return None
        return txt

    # ---------- Step 1: find CI entrypoints ----------
    entrypoints: List[str] = []

    for ep in CI_ENTRYPOINTS_EXACT:
        if ep in path_to_sha:
            entrypoints.append(ep)

    for p in path_to_sha.keys():
        if p.startswith(GHA_WORKFLOWS_DIR.rstrip("/") + "/") and p.lower().endswith((".yml", ".yaml")):
            entrypoints.append(p)

    entrypoints = sorted(set(entrypoints))

    # ---------- Step 2: always gather gradle files ----------
    gradle_files: List[str] = []
    for p in path_to_sha.keys():
        if is_gradle(p):
            gradle_files.append(p)
    gradle_files = sorted(set(gradle_files))

    # ---------- Step 3: workflow expansion ----------
    scanned_paths: Set[str] = set()
    scanned_text_blocks: List[Tuple[str, str]] = []

    unresolved_dynamic_refs_count = 0
    unresolved_dynamic_samples: List[str] = []

    q: List[Tuple[str, int]] = []
    for ep in entrypoints:
        q.append((ep, 0))

    cat = {
        "entrypoints": set(entrypoints),
        "local_actions": set(),
        "called_files": set(),
        "gradle": set(gradle_files),
    }

    def enqueue_path(p: str, depth: int, category: str) -> None:
        nonlocal q
        if len(scanned_paths) >= EXPAND_MAX_FILES:
            return
        p = p.replace("\\", "/").lstrip("/")
        if p in scanned_paths:
            return
        if p not in path_to_sha:
            return
        scanned_paths.add(p)
        cat.get(category, set()).add(p)
        q.append((p, depth))

    for gf in gradle_files:
        enqueue_path(gf, 0, "gradle")

    while q and len(scanned_text_blocks) < EXPAND_MAX_FILES:
        p, depth = q.pop(0)
        txt = read_path(p)
        if txt is None:
            continue

        # normalize
        norm = txt
        if is_gradle(p):
            norm = strip_comments_gradle(norm)
            norm = pre_sanitize_for_triggers(norm)
        else:
            norm = strip_comments(norm)
            norm = normalize_block_keys(norm)
            norm = strip_irrelevant_ci_lines(norm)
            norm = pre_sanitize_for_triggers(norm)

        scanned_text_blocks.append((p, norm))

        if depth < EXPAND_MAX_DEPTH:
            la_dirs, file_refs, unr_cnt, unr_samples = extract_references_from_text(txt, p)
            unresolved_dynamic_refs_count += unr_cnt
            for s in unr_samples:
                if len(unresolved_dynamic_samples) < 20 and s not in unresolved_dynamic_samples:
                    unresolved_dynamic_samples.append(s)

            for d in sorted(la_dirs):
                for cand_dir in _candidate_paths_for_ref(d, p):
                    for am in _resolve_local_action_metadata(cand_dir):
                        if am in path_to_sha:
                            cat["local_actions"].add(cand_dir)
                            enqueue_path(am, depth + 1, "called_files")

            for r in sorted(file_refs):
                for cand in _candidate_paths_for_ref(r, p):
                    if cand in path_to_sha:
                        enqueue_path(cand, depth + 1, "called_files")

        if sum(len(t) for _, t in scanned_text_blocks) > EXPAND_MAX_TEXT_CHARS:
            break

    # ---------- Step 4: compute features & styles ----------
    features = {
        "yaml_loc": 0,
        "gradle_loc": 0,
        "runs_on": set(),
        "matrix_width_hint": 0,
        "api_levels": set(),
        "reliability_flags": set(),
        "gmd_config_present": False,
        "gmd_identifiers": set(),
    }

    RX_RUNS_ON = re.compile(r"(?mi)^\s*runs-on\s*:\s*(.+)$")
    RX_LIST_ITEM = re.compile(r"(?mi)^\s*-\s")
    RX_API_LEVEL = re.compile(r"(?i)\bandroid[-_ ]?(\d{2})\b|system-images;android-(\d{2})\b")
    RX_RETRY = re.compile(r"(?i)\bretry\b|\bmax-attempts\b|\bflaky\b")
    RX_TIMEOUT = re.compile(r"(?i)\btimeout[- ]?minutes\b|\btimeout\b")
    RX_HEADLESS = re.compile(r"(?i)\bheadless\b|\b-no-window\b|\b-no-boot-anim\b")
    RX_WAIT_FOR_DEVICE = re.compile(r"(?i)\badb\s+wait[- ]?for[- ]?device\b")

    GMD_BLOCK_NAME_RX = re.compile(
        r'(\w+)\s*\(\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
        re.I
    )

    has_gradle_anywhere_repo = False

    for path, norm in scanned_text_blocks:
        if is_yaml(path):
            features["yaml_loc"] += len(norm.splitlines())
            for m in RX_RUNS_ON.finditer(norm):
                features["runs_on"].add((m.group(1) or "").strip())
            features["matrix_width_hint"] = max(features["matrix_width_hint"], len(RX_LIST_ITEM.findall(norm)))

        if is_gradle(path):
            features["gradle_loc"] += len(norm.splitlines())
            if not features["gmd_config_present"] and find_gmd_block(norm):
                features["gmd_config_present"] = True
                for m in GMD_BLOCK_NAME_RX.finditer(norm):
                    ident = (m.group(1) or "").strip()
                    if ident:
                        features["gmd_identifiers"].add(ident)

        for m in RX_API_LEVEL.finditer(norm):
            for g in m.groups():
                if g and g.isdigit():
                    features["api_levels"].add(int(g))
        if RX_RETRY.search(norm):
            features["reliability_flags"].add("retry")
        if RX_TIMEOUT.search(norm):
            features["reliability_flags"].add("timeout")
        if RX_HEADLESS.search(norm):
            features["reliability_flags"].add("headless")
        if RX_WAIT_FOR_DEVICE.search(norm):
            features["reliability_flags"].add("wait_for_device")

        if GRADLE_ANYWHERE_RE.search(norm) or GRADLE_BUILD_ACTION_RE.search(norm):
            has_gradle_anywhere_repo = True

    trigger_labels: List[str] = []
    trigger_groups: List[str] = []
    device_labels: List[str] = []
    device_groups: List[str] = []

    # IMPORTANT: single-pass per block so Flutter gating can use same-block runtime evidence
    for _path, norm in scanned_text_blocks:
        low = (norm or "").lower()

        # device signals
        d_labels, d_groups = collect_hits_with_groups(DEVICE_PATTERNS, low)

        # trigger signals
        t_labels, t_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, low)
        fb_labels, fb_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, low)
        if fb_labels:
            t_labels = unique_preserve(t_labels + fb_labels)
            t_groups = unique_preserve(t_groups + fb_groups)

        # GHA Gradle action inputs
        if any_match(GHA_GRADLE_INPUTS, norm):
            gha_tied = re.search(
                r"(?mi)^\s*uses\s*:\s*(reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd)@",
                norm,
            )
            if has_gradle_anywhere_repo or gha_tied:
                if "gha gradle inputs/script" not in t_labels:
                    t_labels.append("gha gradle inputs/script")
                if "Gradle" not in t_groups:
                    t_groups.append("Gradle")

        # NEW: GHA GMD inputs (align with YAML scanner)
        if any_match(GHA_GMD_INPUTS, norm):
            if "variant/device AndroidTest" not in t_labels:
                t_labels.append("variant/device AndroidTest")
            if "Gradle" not in t_groups:
                t_groups.append("Gradle")

        # NEW: STRICT Flutter gating (align with YAML scanner)
        if flutter_androidish_in_block(norm, d_labels, d_groups):
            if "flutter integration test" not in t_labels:
                t_labels.append("flutter integration test")
            if "Flutter" not in t_groups:
                t_groups.append("Flutter")

        # Filter weak "other gha emulator" when no Android context and no test triggers
        if ("other gha emulator" in d_labels) and (not ANDROID_CONTEXT_RE.search(low)) and (not t_labels):
            d_labels = [l for l in d_labels if l != "other gha emulator"]
            if not d_labels:
                d_groups = [g for g in d_groups if g != "Emulator"]

        trigger_labels = unique_preserve(trigger_labels + t_labels)
        trigger_groups = unique_preserve(trigger_groups + t_groups)
        device_labels = unique_preserve(device_labels + d_labels)
        device_groups = unique_preserve(device_groups + d_groups)

    has_test_trigger = bool(trigger_labels)

    strong_seen = bool(set(device_labels) & STRONG_DEVICE_LABELS)
    if not (strong_seen or has_test_trigger):
        device_labels = [l for l in device_labels if l in STRONG_DEVICE_LABELS]
        if not device_labels:
            device_groups = []

    # Provider/env declarations
    repo_text = "\n".join([t for _, t in scanned_text_blocks])
    provider = detect_provider(repo_text)
    inline_decl = has_inline_env(repo_text)
    config_decl = has_config_env(repo_text)

    # NOTE: Removed the BrowserStack “drop unless supported” guard to align with YAML scanner behavior.

    styles: Set[str] = set()

    if "Third_Party_Lab" in device_groups:
        styles.add("ThirdParty")

    if "Emulator" in device_groups:
        lbls = set(device_labels)
        # UPDATED: your requested mapping
        if lbls & EMULATOR_COMMUNITY_LABELS:
            styles.add("Emu_Community")
        if lbls & EMULATOR_CUSTOM_LABELS:
            styles.add("Emu_Custom")

    def has_gmd_gradle_trigger(tlabs: List[str]) -> bool:
        L = {l.lower() for l in tlabs}
        if any("manageddevice androidtest" in l for l in L):
            return True
        if ("variant/device androidtest" in L) and not any(x in L for x in {"connected.*android.*", "connectedandroidtest", "connectedbenchmarkandroidtest", "spoon", "marathon"}):
            return True
        return False

    if features["gmd_config_present"] or has_gmd_gradle_trigger(trigger_labels):
        styles.add("GMD")

    meta = {
        "procedure": {
            "timeline_scope": TIMELINE_SCOPE,
            "expanded_refs": True,
            "expand_max_depth": EXPAND_MAX_DEPTH,
            "expand_max_files": EXPAND_MAX_FILES,
        },
        "expansion": {
            "entrypoints_count": len(entrypoints),
            "gradle_files_count": len(gradle_files),
            "scanned_files_count": len(scanned_text_blocks),
            "called_files_count": len(cat["called_files"]),
            "local_action_dirs_count": len(cat["local_actions"]),
            "unresolved_dynamic_refs_count": int(unresolved_dynamic_refs_count),
            "unresolved_dynamic_samples": unresolved_dynamic_samples[:20],
            "scanned_paths_sample": sorted([p for p, _ in scanned_text_blocks])[:50],
        },
        "features": {
            "yaml_loc": features["yaml_loc"],
            "gradle_loc": features["gradle_loc"],
            "runs_on": sorted(features["runs_on"]),
            "matrix_width_hint": features["matrix_width_hint"],
            "api_levels": sorted(int(x) for x in features["api_levels"]),
            "reliability_flags": sorted(features["reliability_flags"]),
            "trigger_labels": sorted(trigger_labels),
            "device_labels": sorted(device_labels),
            "provider": provider,
            "inline_env": bool(inline_decl),
            "config_env": bool(config_decl),
            "gradle_present_repo": bool(has_gradle_anywhere_repo),
            "gmd_config_present": bool(features["gmd_config_present"]),
            "gmd_identifiers": sorted(features["gmd_identifiers"]) if features["gmd_identifiers"] else [],
        },
    }
    return styles, meta

# ===========================
# Timeline building (DEFAULT branch ONLY)
# ===========================
def empty_result(repo: Path, cutoff_iso: str) -> Dict:
    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "default_branch_ref": default_branch_ref(repo),
        "cutoff_default_head": None,
        "first_commit_date": None,
        "timeline_scope": TIMELINE_SCOPE,
        "timeline": [],
        "events": {"Emu_Community": [], "Emu_Custom": [], "GMD": [], "ThirdParty": []},
        "snapshot_as_of_cutoff_default": [],
        "empty_gaps": [],
        "qa_issue": "No default-branch head at cutoff or no detectable environment at cutoff",
    }

def build_timeline(repo: Path, cutoff_iso: str) -> Dict:
    ref = default_branch_ref(repo)
    head = cutoff_head_default(repo, cutoff_iso)
    if not head:
        return empty_result(repo, cutoff_iso)

    commits = commits_touching_relevant_default(repo, head)

    commit_dates: Dict[str, str] = {}
    for c in commits:
        d = get_commit_date_iso(repo, c)
        if d:
            commit_dates[c] = d
    first_commit_date = commit_dates.get(commits[0]) if commits else None

    timeline: List[Dict] = []
    events: Dict[str, List[Dict]] = {"Emu_Community": [], "Emu_Custom": [], "GMD": [], "ThirdParty": []}
    prev: Set[str] = set()

    empty_gaps: List[Dict] = []
    gap_open: Optional[Dict] = None

    root = earliest_commit_first_parent(repo, head)
    if root:
        base_styles, base_meta = scan_treeish_for_styles_expanded(repo, root)
        if base_styles:
            baseline_date = get_commit_date_iso(repo, root)
            timeline.append({"date": baseline_date, "commit": root, "styles": sorted(base_styles), **base_meta})
            for s in sorted(base_styles):
                if s in events:
                    events[s].append({"event": "added", "date": baseline_date, "commit": root})
            prev = set(base_styles)

    for c in commits:
        styles_now, meta_now = scan_treeish_for_styles_expanded(repo, c)
        dt = commit_dates.get(c)

        if SUPPRESS_EMPTY_ROWS and not styles_now:
            if CAPTURE_EMPTY_GAPS and gap_open is None:
                gap_open = {"start_date": dt, "start_commit": c, "prev_state": "+".join(sorted(prev)) if prev else ""}
            continue

        if CAPTURE_EMPTY_GAPS and gap_open is not None and styles_now:
            gap_open["end_date"] = dt
            gap_open["end_commit"] = c
            gap_open["next_state"] = "+".join(sorted(styles_now))
            try:
                d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)
                d2 = datetime.fromisoformat((dt or "").replace("Z", "+00:00")).replace(tzinfo=None)
                gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
            except Exception:
                gap_open["duration_days"] = None
            gap_open["censored"] = False
            empty_gaps.append(gap_open)
            gap_open = None

        if styles_now != prev:
            if EMIT_NONE_STATE and not styles_now:
                timeline.append({"date": dt, "commit": c, "styles": [], **meta_now})
            elif styles_now:
                timeline.append({"date": dt, "commit": c, "styles": sorted(styles_now), **meta_now})

            added = styles_now - prev
            removed = prev - styles_now
            for s in sorted(added):
                if s in events:
                    events[s].append({"event": "added", "date": dt, "commit": c})
            for s in sorted(removed):
                if s in events:
                    events[s].append({"event": "removed", "date": dt, "commit": c})
            prev = styles_now
        else:
            if styles_now:
                for s in sorted(styles_now):
                    if s in events:
                        events[s].append({"event": "maintenance", "date": dt, "commit": c})

    if CAPTURE_EMPTY_GAPS and gap_open is not None:
        gap_open["end_date"] = cutoff_iso
        gap_open["end_commit"] = head
        gap_open["next_state"] = ""
        try:
            d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)
            d2 = datetime.fromisoformat(cutoff_iso.replace("Z", "+00:00")).replace(tzinfo=None)
            gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
        except Exception:
            gap_open["duration_days"] = None
        gap_open["censored"] = True
        empty_gaps.append(gap_open)
        gap_open = None

    snap_styles, snap_meta = scan_treeish_for_styles_expanded(repo, head)
    snapshot_default = sorted(snap_styles)

    qa_issue = None
    if not snapshot_default:
        qa_issue = "No environment detected at cutoff (default branch) under expanded workflow procedure"

    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "default_branch_ref": ref,
        "cutoff_default_head": head,
        "first_commit_date": first_commit_date,
        "timeline_scope": TIMELINE_SCOPE,
        "timeline": timeline,
        "events": events,
        "snapshot_as_of_cutoff_default": snapshot_default,
        "snapshot_meta_at_cutoff_default": snap_meta,
        "empty_gaps": empty_gaps,
        **({"qa_issue": qa_issue} if qa_issue else {}),
    }

# ===========================
# Runner + index/summary
# ===========================
def output_path_for(repo: Path) -> Path:
    return OUTPUT_MINE / f"{repo.name}.emulator_timeline.default_refs.json"

def list_repos_once(root: Path) -> List[Path]:
    rs = list_repos(root)
    rs.sort(key=lambda p: p.name.lower())
    return rs

def iso_min(dts: List[str]) -> Optional[str]:
    ds = [d for d in dts if d]
    return min(ds) if ds else None

def derive_summary_fields(rec: Dict) -> Dict:
    name = rec.get("repo_name")
    snap_def = rec.get("snapshot_as_of_cutoff_default", [])
    events = rec.get("events", {})
    first_events: List[Optional[str]] = []
    for k in ("Emu_Community", "Emu_Custom", "GMD", "ThirdParty"):
        for e in events.get(k, []):
            if e.get("event") == "added":
                first_events.append(e.get("date"))
    first_env_date = iso_min([d for d in first_events if d])
    transitions = 0
    for k in events:
        transitions += sum(1 for e in events[k] if e.get("event") in ("added", "removed"))
    gaps = rec.get("empty_gaps", [])
    gap_days = 0.0
    for g in gaps:
        try:
            gap_days += float(g.get("duration_days") or 0.0)
        except Exception:
            pass

    cutoff_meta = rec.get("snapshot_meta_at_cutoff_default", {}) or {}
    exp = (cutoff_meta.get("expansion", {}) or {})

    return {
        "repo_name": name,
        "default_branch_ref": rec.get("default_branch_ref", ""),
        "has_env_at_cutoff_default": 1 if snap_def else 0,
        "cutoff_states_default": "+".join(snap_def),
        "first_env_date": first_env_date or "",
        "transitions": transitions,
        "total_gap_days": round(gap_days, 2),
        "cutoff_scanned_files": exp.get("scanned_files_count", ""),
        "cutoff_called_files": exp.get("called_files_count", ""),
        "cutoff_unresolved_dynamic_refs": exp.get("unresolved_dynamic_refs_count", ""),
        "qa_issue": rec.get("qa_issue", ""),
    }

def run_miner(cutoff_iso: str = CUTOFF_ISO, max_repos: int = MAX_REPOS) -> None:
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

    repos = list_repos_once(INPUT_CLONES)
    if max_repos and max_repos > 0:
        repos = repos[:max_repos]

    if not repos:
        print(f"[info] No git repos found under: {INPUT_CLONES}")
        return

    print(f"[info] Work root:         {WORK_ROOT}")
    print(f"[info] Input repos:       {INPUT_CLONES}")
    print(f"[info] Output JSON dir:   {OUTPUT_MINE}")
    print(f"[info] Found repos:       {len(repos)}")
    print(f"[info] Cutoff (UTC):      {cutoff_iso}")
    print(f"[info] Timeline scope:    {TIMELINE_SCOPE}")
    print(f"[info] Expansion bounds:  depth={EXPAND_MAX_DEPTH}, max_files={EXPAND_MAX_FILES}, max_text_chars={EXPAND_MAX_TEXT_CHARS}")
    print(f"[info] Options: suppress_empty_rows={SUPPRESS_EMPTY_ROWS}, capture_empty_gaps={CAPTURE_EMPTY_GAPS}, emit_none={EMIT_NONE_STATE}, blob_max_size={BLOB_MAX_SIZE}")

    written = skipped = errors = 0
    summaries: List[Dict] = []

    def process(repo: Path) -> Tuple[str, Path, Optional[Dict], Optional[str]]:
        try:
            outp = output_path_for(repo)
            if RESUME_IF_EXISTS and outp.exists():
                return ("skipped", repo, None, None)
            data = build_timeline(repo, cutoff_iso)
            with outp.open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            return ("written", repo, data, None)
        except Exception as e:
            return ("error", repo, None, str(e))

    with PoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(process, r): r for r in repos}
        total = len(futs)
        for i, fut in enumerate(as_completed(futs), 1):
            repo = futs[fut]
            rel = repo.relative_to(INPUT_CLONES) if repo != INPUT_CLONES else Path(repo.name)
            status, _repo, data, err = fut.result()
            if status == "written":
                print(f"[{i}/{total}] Wrote: {rel}")
                written += 1
                summaries.append(derive_summary_fields(data))  # type: ignore[arg-type]
            elif status == "skipped":
                print(f"[{i}/{total}] Skipped (exists): {rel}")
                skipped += 1
                try:
                    with output_path_for(repo).open("r", encoding="utf-8") as f:
                        data2 = json.load(f)
                    summaries.append(derive_summary_fields(data2))
                except Exception:
                    pass
            else:
                print(f"[{i}/{total}] [error] {rel}: {err}", file=sys.stderr)
                errors += 1

    idx_json = OUTPUT_MINE / "index_summary.default_refs.json"
    with idx_json.open("w", encoding="utf-8") as f:
        json.dump({"cutoff": cutoff_iso, "repos": summaries}, f, ensure_ascii=False, indent=2)

    idx_csv = OUTPUT_MINE / "index_summary.default_refs.csv"
    with idx_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "repo_name",
                "default_branch_ref",
                "has_env_at_cutoff_default",
                "cutoff_states_default",
                "first_env_date",
                "transitions",
                "total_gap_days",
                "cutoff_scanned_files",
                "cutoff_called_files",
                "cutoff_unresolved_dynamic_refs",
                "qa_issue",
            ],
        )
        w.writeheader()
        for row in summaries:
            w.writerow(row)

    qa_csv = OUTPUT_MINE / "qa_issues.default_refs.csv"
    with qa_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["repo_name", "qa_issue"])
        w.writeheader()
        for row in summaries:
            if row.get("qa_issue"):
                w.writerow({"repo_name": row["repo_name"], "qa_issue": row["qa_issue"]})

    print(f"[done] JSON -> {OUTPUT_MINE}  (written={written}, skipped={skipped}, errors={errors})")
    print(f"[done] Summary: {idx_csv.name}, {idx_json.name}, QA: {qa_csv.name}")

if __name__ == "__main__":
    _failfast_checks()
    print("[run] Starting miner (default branch + expanded refs)…")
    run_miner(cutoff_iso=CUTOFF_ISO, max_repos=MAX_REPOS)
    print("[run] Finished.")


[run] Starting miner (default branch + expanded refs)…
[info] Work root:         C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2
[info] Input repos:       C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone
[info] Output JSON dir:   C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Mine
[info] Found repos:       12
[info] Cutoff (UTC):      2025-08-10 23:59:59 +0000
[info] Timeline scope:    default_first_parent
[info] Expansion bounds:  depth=3, max_files=250, max_text_chars=5000000
[info] Options: suppress_empty_rows=True, capture_empty_gaps=True, emit_none=False, blob_max_size=2000000
[1/12] Wrote: haiyangwu__mediasoup-client-android
[2/12] Wrote: kickstarter__android-oss
